# Event study: how markets react to U.S. CPI and payroll surprises (PUBLIC DATA)

Question: on the day of a CPI or payroll release, do the 2-year Treasury yield, the dollar and the S&P 500 move with the size of the surprise?

- **Releases**: every CPI and Employment Situation report since January 2016, dated with ALFRED vintages.
- **Surprise**: the first print minus a *model forecast (illustrative)*, the average of the three previous prints as they stood the day before. No consensus is used: there is no free, reliable public source.
- **Market move**: daily close-to-close change on the release day (FRED).
- **Limits**: daily data (other news the same day), a model instead of a consensus (noisier surprise), COVID months excluded.

In [1]:
import sys, pandas as pd, numpy as np
sys.path.insert(0, '../src')
import event_study as es
pd.set_option('display.precision', 2)
t = es.release_table()
t.groupby('indicator').agg(releases=('ref', 'size'), first=('release_date', 'min'), last=('release_date', 'max'))

,releases,first,last
indicator,,,
CPI,129,2016-02-19,2026-11-10
NFP,130,2016-02-05,2026-11-06


## 1. What a surprise looks like
Core CPI: published month-on-month change (one decimal) minus the model. Payrolls: monthly change (thousands) minus the model.

In [2]:
t.groupby('indicator').surprise.describe()

,count,mean,std,min,25%,50%,75%,max
indicator,,,,,,,,
CPI,129.0,-2.34e-04,0.18,-0.55,-0.07,0.02,0.09,0.74
NFP,130.0,-4.32e+00,2308.27,-20429.33,-99.58,-19.50,64.75,11317.00


In [3]:
cols = ['ref', 'release_date', 'core_actual', 'core_model', 'z', 'y2_bp', 'usd_pct', 'eq_pct']
t[t.indicator == 'CPI'].sort_values('z').iloc[[0, 1, 2, -3, -2, -1]][cols]

,ref,release_date,core_actual,core_model,z,y2_bp,usd_pct,eq_pct
67,2021-08-01,2021-09-14,0.1,0.65,-3.48,0.0,-0.02,-0.45
66,2021-07-01,2021-08-11,0.3,0.84,-3.46,-1.0,-0.19,-0.16
51,2020-04-01,2020-05-12,-0.4,0.12,-3.31,0.0,-0.14,-2.06
53,2020-06-01,2020-07-14,0.2,-0.20,2.56,-2.0,0.17,0.94
54,2020-07-01,2020-08-12,0.6,-0.09,4.39,0.0,-0.10,2.13
63,2021-04-01,2021-05-12,0.9,0.16,4.72,0.0,0.61,-2.67


## 2. Regressions: move on the day = a + b × surprise (in standard deviations)
OLS with HC1 robust standard errors, reference months Mar 2020 - Mar 2021 excluded.

In [4]:
r = es.regressions(t)
r

,indicator,market,n,beta,ci_low,ci_high,p_value,r2,n_big,same_sign_big
0,CPI,y2_bp,115,2.91,1.11,4.71e+00,1.51e-03,0.12,22,0.73
1,CPI,y10_bp,115,2.30,1.11,3.49e+00,1.57e-04,0.11,22,0.73
2,CPI,usd_pct,116,0.16,0.09,2.23e-01,6.61e-06,0.15,23,0.74
3,CPI,eq_pct,115,-0.31,-0.61,-1.48e-03,4.89e-02,0.04,22,0.41
4,NFP,y2_bp,117,2.39,0.82,3.96e+00,2.84e-03,0.07,28,0.71
5,NFP,y10_bp,117,1.37,0.04,2.70e+00,4.42e-02,0.04,28,0.57
6,NFP,usd_pct,117,0.11,0.03,1.80e-01,3.92e-03,0.09,28,0.57
7,NFP,eq_pct,115,0.23,-0.02,4.69e-01,7.04e-02,0.02,27,0.52


## 3. Are release days more volatile than other days?
Average absolute daily move on event days divided by the average on all other days since 2016.

In [5]:
v = es.volatility(t)
v.pivot(index='group', columns='market', values='ratio')

market,eq_pct,usd_pct,y10_bp,y2_bp
group,,,,
CPI days,1.07,1.25,1.33,1.63
FOMC days,1.35,0.74,1.37,2.13
Payroll days,1.22,1.23,1.45,1.78


## 4. Robustness: surprise measured against the previous print instead of the model

In [6]:
import statsmodels.api as sm
d = t[(t.indicator == 'CPI') & ~t.covid].copy()
d['s_prev'] = d.core_actual - d.core_prev
d['z_prev'] = d.s_prev / d.s_prev.std()
dd = d[['z_prev', 'y2_bp']].dropna()
fit = sm.OLS(dd.y2_bp, sm.add_constant(dd.z_prev)).fit(cov_type='HC1')
print(f"core CPI vs previous print: {fit.params['z_prev']:+.2f} bp per sd, 95% CI "
      f"{fit.conf_int().loc['z_prev', 0]:+.2f} to {fit.conf_int().loc['z_prev', 1]:+.2f}, n={len(dd)}")

core CPI vs previous print: +2.63 bp per sd, 95% CI +1.05 to +4.20, n=115


In [7]:
d = t[(t.indicator == 'NFP')].copy()
d['rev1'] = d.nfp_prev_revised - d.nfp_prev_first
print(f"payrolls: previous month revised by {d.rev1.abs().mean():.0f}K on average (absolute), "
      f"{(d.rev1.abs() >= 25).mean():.0%} of releases revised it by 25K or more")

payrolls: previous month revised by 40K on average (absolute), 51% of releases revised it by 25K or more


## 5. Reading
- The 2-year yield is the market that moves most with the surprise: it prices the path of Fed rates.
- The effect is real but noisy on daily data: R² is low because other news moves markets the same day.
- A true consensus would give larger and cleaner slopes; the model surprise is a conservative lower bound.
- Revisions to the previous month are large enough that 'previous' must always be the revised figure.